In [ ]:
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    recall_score, classification_report, confusion_matrix,
    f1_score, roc_auc_score, make_scorer
)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Load data
X_train = joblib.load("../data/processed/X_train.pkl")
X_test = joblib.load("../data/processed/X_test.pkl")
y_train = joblib.load("../data/processed/y_train.pkl")
y_test = joblib.load("../data/processed/y_test.pkl")

# Focus on recall of the positive class (churn = 1)
recall_scorer = make_scorer(recall_score, pos_label=1, greater_is_better=True)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
    ]),
    "RandomForest": RandomForestClassifier(random_state=42, class_weight="balanced"),
    "DecisionTree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model",SVC(class_weight="balanced"))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),
    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    )
}

param_grids = {
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1, 5, 10],
        "model__solver": ["liblinear", "lbfgs"]
    },
    "RandomForest": {
        "n_estimators": [200, 500],
        "max_depth": [None, 8, 12, 16],
        "min_samples_leaf": [1, 3, 5],
        "max_features": ["sqrt", "log2", None]
    },
    "DecisionTree": {
        "max_depth": [None, 8, 12, 16],
        "min_samples_leaf": [1, 3, 5],
        "max_features": ["sqrt", "log2", None]
    },
    "SVM": {
        "model__C": [0.1, 1, 5, 10],
        "model__gamma": ["scale", "auto", 0.01, 0.1],
        "model__kernel": ["rbf", "linear"]
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 11, 15],
        "model__weights": ["uniform", "distance"]
    },
    "XGBoost": {
        "n_estimators": [200, 500],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "scale_pos_weight": [1, 2, 4, 6]
    }
}

results = []

for name, model in models.items():
    print(f"\n=== Tuning {name} ===")
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        scoring=recall_scorer,
        cv=cv,
        n_jobs=-1,
        verbose=0
    )

    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, "predict_proba") else None

    recall = recall_score(y_test, y_pred, pos_label=1)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    auc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

    results.append({
        "Model": name,
        "Best_Params": grid.best_params_,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": auc
    })

    print("Best params:", grid.best_params_)
    print("Recall:", recall)
    print("F1:", f1)
    print("ROC AUC:", auc)
    print(classification_report(y_test, y_pred))

results_df = pd.DataFrame(results).sort_values("Recall", ascending=False)
print("\n=== Final ranking by Recall ===")
print(results_df)

best_model_name = results_df.iloc[0]["Model"]
print(f"\nBest model by Recall: {best_model_name}")


=== Tuning LogisticRegression ===
Best params: {'model__C': 0.01, 'model__solver': 'liblinear'}
Recall: 0.8145161290322581
F1: 0.6133603238866396
ROC AUC: 0.8481898427172136
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.49      0.81      0.61       372

    accuracy                           0.73      1405
   macro avg       0.70      0.76      0.70      1405
weighted avg       0.80      0.73      0.74      1405


=== Tuning RandomForest ===
Best params: {'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 500}
Recall: 0.803763440860215
F1: 0.6261780104712041
ROC AUC: 0.8515676232707741
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1033
           1       0.51      0.80      0.63       372

    accuracy                           0.75      1405
   macro avg       0.71      0.76      0.72      1405
weighted avg       0.8